# Mixed-Precision Training

> At step 3000, the loss suddenly becomes NaN with little explanation in the logs. One common source of trouble is FP16's limited range. Its smallest positive normal value is about `6.1e-5`. Smaller values enter the subnormal range, with reduced precision, and sufficiently small values underflow to zero. Small training gradients can encounter this limit.
>
> Lower precision can save memory and accelerate computation. Mixed-Precision Training combines those benefits with operations kept at higher precision for stability.
>
> This appendix covers four groups of concepts:
>
> 1. **Floating-point representation:** Exponent bits determine range, mantissa bits determine precision. FP16, BF16, FP8, and FP4 allocate them differently.
>
> 2. **Mixed precision:** Use lower precision for matrix multiplication, while gradient accumulation, optimizer states, and sensitive normalizations retain higher precision.
>
> 3. **Stability:** Loss Scaling, FP32 master weights, and dtype selection address underflow, accumulated rounding, and limited range.
>
> 4. **Precision advances:** FP16 and BF16 lead to FP8 and fine-grained scaling, QLoRA's NF4, and BitNet's 1.58-bit weights.


Why not use low precision everywhere? Adding `0.00001` to `1.0` can lose the change in FP16 because its mantissa is too short. Large matrix multiplications can nevertheless tolerate some rounding error, particularly with higher-precision accumulation; errors do not always cancel.

Training therefore selects precision by operation. FP16, BF16, or FP8 can accelerate matmuls, while gradient accumulation, optimizer states, and some normalization operations use FP32.

This is **Mixed-Precision Training**. We first examine sign, exponent, and mantissa bits, then identify what Loss Scaling and FP32 master weights each protect.


## 1. Floating-Point Bit Layout

A floating-point number consists of a sign bit, exponent bits, and mantissa bits. Together they determine how large and how accurately a number can be represented.

- **Sign:** One bit; 0 is positive, 1 negative.
- **Exponent:** Determines dynamic range, spanning large and small magnitudes.
- **Mantissa:** Determines precision, distinguishing nearby values at the same magnitude.

Both exponent and mantissa matter. With a fixed bit budget, another exponent bit means one fewer mantissa bit: wider range trades against precision.

BF16 makes this trade explicitly. Like FP16 it uses 16 bits, but transfers three mantissa bits to the exponent, gaining range at the cost of precision. The examples below show why this can help training.


The common formats allocate bits as follows:

| dtype | Bits | Sign | Exponent | Mantissa | Notes |
|:---|:---|:---|:---|:---|:---|
| FP32 | 32 | 1 | 8 | 23 | Common master-weight and optimizer-state precision |
| FP16 | 16 | 1 | 5 | 10 | Limited range, max≈65504; often needs Loss Scaling |
| BF16 | 16 | 1 | 8 | 7 | FP32-like exponent range, fewer mantissa bits |
| FP8 E4M3 | 8 | 1 | 4 | 3 | Higher precision of the two FP8 variants |
| FP8 E5M2 | 8 | 1 | 5 | 2 | Wider range of the two FP8 variants |
| FP4 E2M1 | 4 | 1 | 2 | 1 | Lower precision; training requires a suitable recipe |
| INT8 | 8 | — | — | — | Integer encoding, with external scale for real values |
| INT4 | 4 | — | — | — | Integer encoding, with external scale for real values |

FP16 and BF16 share a 16-bit budget but distribute it differently. E4M3 and E5M2 make the same range-versus-precision trade at eight bits. Integer formats have no floating-point exponent or mantissa.


### 1.1 Dynamic Range and Precision

Exponent bits determine the available range, together with the bias and reserved encodings. FP16 has five exponent bits and a maximum finite value of 65504. BF16 has eight exponent bits and a maximum of about $3.39\times10^{38}$, close to FP32's $3.40\times10^{38}$.

Mantissa bits determine spacing. Near 1, FP16's ten bits give spacing $2^{-10}\approx0.001$; BF16's seven give $2^{-7}\approx0.008$. BF16's spacing is eight times larger at that scale. These are spacing estimates, not a fixed relative error for every value.

Gradients can span $10^{-4}$ to $10^{-7}$, or other ranges depending on the model. BF16 represents these magnitudes readily. FP16's smallest **normal** positive number is about $6.1\times10^{-5}$, but it can still represent subnormals down to roughly $5.96\times10^{-8}$. Subnormals lose relative precision; still smaller values round to zero, and some kernels may flush subnormals.

A coarse nonzero gradient can be more useful than a zeroed gradient. BF16's range therefore removes much of FP16's underflow pressure, although precision and other sources of instability still matter.


In [ ]:
# Inspect the actual range and precision differences with torch dtypes
import torch

dtypes = [torch.float32, torch.float16, torch.bfloat16]

print('=== Maximum representable value (dynamic range upper bound) ===')
for dt in dtypes:
    # torch.finfo returns the numeric info for the dtype
    info = torch.finfo(dt)
    print(f'{str(dt):<20} max = {info.max:<25.6e}  min = {info.min:.6e}')

print()
print('=== Smallest positive normal number (underflow threshold; below this it becomes 0) ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} tiny = {info.tiny:<25.6e}')

print()
print('=== Relative precision (smallest relative gap between two adjacent numbers) ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} eps = {info.eps:<25.6e}  (about {info.eps:.4f})')

print()
print('Key observations:')
print('  FP32 vs BF16 max is nearly identical (both reach 3.4e+38) — same range.')
print('  FP16 max is only 65504 — slightly larger intermediate activations overflow to Inf.')
print('  BF16 eps is 8x larger than FP16 — worse precision, but adequate for training.')


### 1.2 Representing 0.1 at Different Precisions

Inspect the actual stored value of 0.1 in several dtypes. It is finite in decimal, but has a repeating binary expansion, so finite binary floating-point formats store an approximation. The difference after a round trip makes precision loss visible.


In [ ]:
# Store the same 0.1 under different precisions, then read it back, and inspect the error
import torch

x = 0.1
print(f'Original value (Python float, essentially FP64): {x:.20f}')
print()

for dt in [torch.float32, torch.float16, torch.bfloat16]:
    # cast there and back, simulating "store then read back"
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    print(f'{str(dt):<20} stored as {stored:.20f}  error {err:.2e}')

print()
print('Now a scenario more sensitive to training: a small gradient 0.0001')
x = 0.0001
print(f'Original value: {x:.10e}')
print()
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    rel_err = err / x if x != 0 else 0
    print(f'{str(dt):<20} stored as {stored:.10e}  relative error {rel_err:.2e}')

print()
print('Key observation: both FP16 and BF16 can still hold 0.0001 (no underflow), but with different precision.')
print('If the gradient shrinks one more notch to 1e-7, FP16 with tiny=6e-5 cannot hold it and rounds it to 0.')


In [ ]:
# Demonstrate FP16 underflow: small gradients accumulate to 0
import torch

# Simulate gradient accumulation: sum gradients of 8 micro-batches
# each gradient is on the order of 1e-7 (common in real training)
grad_per_step = torch.full((4,), 1e-7, dtype=torch.float32)

# FP32 accumulation
acc_fp32 = torch.zeros(4, dtype=torch.float32)
for _ in range(8):
    acc_fp32 += grad_per_step

# FP16 accumulation (if the gradient itself were stored in FP16)
acc_fp16 = torch.zeros(4, dtype=torch.float16)
grad_fp16 = grad_per_step.to(torch.float16)
print(f'Single gradient as FP16: {grad_fp16}')
print(f'  (note: 1e-7 < FP16 tiny=6e-5, so it rounds to 0 directly)')
for _ in range(8):
    acc_fp16 += grad_fp16

# BF16 accumulation
acc_bf16 = torch.zeros(4, dtype=torch.bfloat16)
grad_bf16 = grad_per_step.to(torch.bfloat16)
print(f'Single gradient as BF16: {grad_bf16}')
for _ in range(8):
    acc_bf16 += grad_bf16

print()
print(f'FP32 accumulation result: {acc_fp32}')
print(f'FP16 accumulation result: {acc_fp16}  <- all zeros, gradient vanished')
print(f'BF16 accumulation result: {acc_bf16}  <- also loses precision, but better than FP16')
print()
print('This is why accumulation in training (gradient accumulation, optimizer state) must stay FP32.')


## 2. FP16 Training

Micikevicius et al.'s 2017 Mixed Precision Training paper explains two important difficulties and their remedies: Loss Scaling and FP32 master weights.

The first is **gradient underflow**. The smallest normal FP16 number is about $6\times10^{-5}$. Some gradients fall below it; values in the subnormal range have reduced precision, and sufficiently small gradients become zero. For example, $10^{-7}$ is subnormal rather than automatically unrepresentable. Losing small gradients removes part of the update signal.

The second is **rounded-away updates**. Mathematically, $1.0+0.0001=1.0001$, but FP16 rounds this back to 1.0. Here the problem is mantissa precision near the weight's magnitude, rather than exponent range. Loss Scaling and master weights address different failure modes.


### 2.1 Loss Scaling

Multiply the loss by a large factor, such as $2^{16}$, before Backward. By the chain rule, gradients grow by the same factor and are less likely to underflow in FP16.

Before updating weights, divide the gradients by that factor:

```text
scaled_loss = loss * S
scaled_loss.backward()       # Scale gradients by S to reduce underflow
for p in params:
    p.grad = p.grad / S       # Restore the unscaled gradient
    p.data -= lr * p.grad     # Update with the restored gradient
```

Here S is the scaling factor. **Fixed scaling** uses a constant, often $2^{16}$, but a large gradient can overflow after scaling.

**Dynamic scaling** checks for Inf or NaN after Backward. If found, it reduces S and skips the optimizer update; it does not automatically rerun the batch. After enough successful steps, it increases S. This adapts the scale to observed gradients. The next examples illustrate scaling and a simplified scale-update policy.


In [ ]:
# Mini demonstration of the effect of loss scaling
import torch

torch.manual_seed(42)
# Simulate a small gradient (common in deep layers during real training)
grad_true = torch.randn(8) * 1e-5  # order of 1e-5
print(f'True gradient (FP32): {grad_true}')
print()

# No scaling, just cast directly to FP16
grad_fp16_noscale = grad_true.to(torch.float16)
print(f'Directly cast to FP16: {grad_fp16_noscale}')
print(f'  Number of non-zero elements: {(grad_fp16_noscale != 0).sum().item()} / 8')
print(f'  -> inspect the nonzero count and error to determine underflow')

print()
# loss scaling: amplify by S=1024, cast to FP16, then divide back
S = 1024
grad_scaled = (grad_true * S).to(torch.float16)
grad_restored = grad_scaled.to(torch.float32) / S
print(f'loss scaling (S={S}):')
print(f'  Amplified then cast to FP16: {grad_scaled}')
print(f'  Divided back by S:          {grad_restored}')
print(f'  Max deviation from truth:   {(grad_true - grad_restored).abs().max().item():.2e}')
print()
print('Compare nonzero counts and restored error; 1e-5 need not underflow to zero.')


In [ ]:
# Minimal implementation of dynamic loss scaling
import torch

class DynamicLossScaler:
    """
    Minimal dynamic loss scaler to illustrate the idea.
    The real implementation is torch.amp.GradScaler.
    """
    def __init__(self, init_scale=2**16, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0

    def scale_loss(self, loss):
        return loss * self.scale

    def unscale_grads(self, grads):
        return [g / self.scale for g in grads]

    def step(self, grads_found_inf):
        """Call after each backward; grads_found_inf indicates whether Inf/NaN was detected"""
        if grads_found_inf:
            self.scale *= self.backoff_factor
            self.steps_since_growth = 0
            print(f'  Inf found, scale halved to {self.scale:.1f}')
        else:
            self.steps_since_growth += 1
            if self.steps_since_growth >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_growth = 0
                print(f'  No Inf for a while, scale doubled to {self.scale:.1f}')


# Demo: simulate several training steps, occasionally injecting Inf
scaler = DynamicLossScaler(init_scale=1024)
inf_events = [False, False, True, False, False, False]  # inject Inf at step 3
for step, has_inf in enumerate(inf_events, 1):
    print(f'step {step}: scale={scaler.scale:.1f}', end='')
    scaler.step(has_inf)
print()
print('Key observation: dynamic scaling backs off automatically on Inf and grows slowly when there is no Inf,')
print('keeping the scale at the edge of "usable" — that is why it is more stable than fixed scaling.')


### 2.2 Operations Retained at FP32

Casting every operation to FP16 is rarely a good training recipe. Operations have different numerical needs:

- **Low-precision candidates:** Matmul and convolution can use Tensor Cores and higher-precision accumulation. Speedup depends on the hardware and workload, rather than a universal 2× factor.
- **Sensitive operations:** Some reductions, normalization, Softmax, and losses benefit from FP32 internal computation because sums, exponentials, and logarithms can amplify numerical problems.

PyTorch's `torch.autocast` chooses per-operation dtypes. Its policies depend on the device and operation; CUDA and CPU do not share one identical list, and output dtype does not always reveal accumulator precision. Consult the [autocast operation reference](https://docs.pytorch.org/docs/stable/amp.html#autocast-op-reference).

The next example rounds logits to FP16 and converts them back to FP32 before Softmax. It isolates **input rounding**, rather than demonstrating a Softmax executed in FP16.


In [ ]:
# Demonstrate FP16 logit rounding followed by FP32 Softmax
import torch

# A set of logits with a large maximum value
torch.manual_seed(0)
logits = torch.randn(4) * 20  # amplify the numeric range
print(f'logits: {logits}')

# FP32 softmax
probs_fp32 = torch.softmax(logits, dim=-1, dtype=torch.float32)
print(f'FP32 softmax: {probs_fp32}')

# Round logits to FP16, then convert back for FP32 Softmax
logits_fp16 = logits.to(torch.float16)
probs_fp16_bad = torch.softmax(logits_fp16.float(), dim=-1)
print(f'FP16-rounded input + FP32 Softmax: {probs_fp16_bad}')

# Compare the error
err = (probs_fp32 - probs_fp16_bad).abs().max().item()
print(f'Max error: {err:.2e}')
print()
print('Softmax computes in FP32 here; the comparison isolates input rounding.')
print('Autocast policies depend on the backend; check the operation reference.')


### 2.3 Common FP16 Problems

Three failure modes are worth separating.

**NaN after an overflow.** An Inf gradient used in an optimizer update can corrupt parameters. A GradScaler checks for nonfinite gradients and skips that update. A suddenly nonfinite loss or parameter histogram helps locate the failure.

**Optimizer-state rounding.** Adam's momentum and variance accumulate information over many steps. FP32 states are a common robust baseline; storing them at lower precision requires a validated optimizer scheme.

**Attention score problems.** $QK^\top$ can overflow or become excessively large under some inputs and scales. Inspect Q/K magnitudes and the $1/\sqrt{d_h}$ factor; use stable Softmax, suitable accumulator precision, or a supported FlashAttention implementation. Longer sequences alone do not guarantee large score magnitudes.

These include both range and precision problems. BF16 widens the range, but cannot cure every failure.


## 3. BF16 Training

BF16 (Brain Float 16) is widely used for LLM training. It offers eight exponent bits like FP32, with only seven mantissa bits.

Its maximum finite value is about $3.39\times10^{38}$ and its smallest normal positive value about $1.18\times10^{-38}$. The exponent range is similar to FP32, while fewer mantissa bits give coarser precision and a slightly different maximum finite value.

Typical weights around 0.01–1.0 and gradients around $10^{-4}$–$10^{-7}$ fit comfortably in this range. BF16 training therefore usually does not need Loss Scaling.

Keep sensitive operations and parameter updates at suitable higher precision. BF16 reduces FP16 overflow/underflow pressure, but Inf and NaN remain possible from unstable computations, data, or optimization. Public reports should be checked before attributing a specific dtype to a model whose recipe is undisclosed.


### 3.1 Dtype Selection with `torch.autocast`

The context manager chooses operation dtypes without requiring a manual cast for every operation:

```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    output = model(inputs)  # Per-operation BF16/FP32 policy
```

Typical candidates include:

- **Lower precision:** `Linear`, `Conv2d`, `matmul`, and `bmm`.
- **Higher precision where listed:** Sensitive normalization, losses, and selected reductions.

The exact list varies by backend. In particular, CPU Softmax on a BF16 input may produce BF16 output; do not assume the CUDA policy applies unchanged. Autocast inserts conversions as needed without changing the model's stored parameter dtype. Conversion overhead should be measured along with matmul gains. The MiniBlock below shows parameter and output dtypes in the available environment.


In [ ]:
# Demonstrate how autocast changes an op's actual dtype
# Both CPU and CUDA support autocast; pick device_type based on the environment
import torch
import torch.nn as nn

class MiniBlock(nn.Module):
    """A mini Transformer block: Linear -> softmax -> Linear"""
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, d)
        self.fc2 = nn.Linear(d, d)

    def forward(self, x):
        # fc1 is a matmul, runs as BF16 under autocast
        h = self.fc1(x)
        # Softmax dtype follows backend policy; CPU and CUDA may differ
        h = torch.softmax(h, dim=-1)
        # fc2 is a matmul again, back to BF16
        return self.fc2(h)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
torch.manual_seed(42)
block = MiniBlock(64).to(device)
x = torch.randn(2, 8, 64, device=device)

# Without autocast: default FP32
print('=== Without autocast ===')
out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}')
print(f'output dtype:     {out.dtype}')

print()
# With BF16 autocast
print('=== With autocast(bfloat16) ===')
with torch.autocast(device_type=dev_type, dtype=torch.bfloat16):
    out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}  <- weights themselves are still FP32')
print(f'output dtype:     {out.dtype}  <- output is BF16')
print()
print('Key observation: autocast does not change the dtype of model weights (still FP32);')
print('it only feeds a cast copy of the weights to matmul ops during forward.')


### 3.2 FP32 Master Weights

Low-precision Forward and Backward do not require low-precision stored weights. A common recipe retains FP32 parameters and computes selected operations with lower-precision copies:

1. Keep FP32 master weights.
2. Cast the operands needed by selected Forward operations to BF16.
3. Backward computes gradients according to the operation's precision policy.
4. Accumulate parameter gradients in FP32 when the stored parameters are FP32.
5. Apply the optimizer update to the FP32 weights.

Why avoid direct BF16 updates? Near $w=1.0$, BF16 spacing is about 0.008, much larger than an update of 0.0001. The small change can disappear when rounded.

Repeated small changes matter: 1000 updates of 0.0001 should change a weight by 0.1, yet direct low-precision updates may repeatedly round away. This demonstrates a risk, rather than claiming every parameter remains at initialization.

With ordinary PyTorch autocast, leave model parameters at FP32 rather than explicitly converting the whole model to BF16. Gradients of FP32 parameters accumulate in FP32; GradScaler controls scaling and overflow checks for FP16, rather than being the mechanism that converts every gradient back to FP32. Other training frameworks can use different master-weight recipes. The next code compares direct BF16 and FP32 accumulation.


In [ ]:
# Demonstrate BF16 master weight accumulation vs FP32 master weight accumulation
import torch

# Simulate 1000 training steps, each with a gradient of 1e-4
n_steps = 1000
lr = 1e-4

# Option A: use BF16 directly as the master weight (wrong)
w_bf16 = torch.tensor([1.0], dtype=torch.bfloat16)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.bfloat16)
    w_bf16 = w_bf16 - grad  # BF16 accumulation

# Option B: FP32 master weight, BF16 only used for forward (correct)
w_fp32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.float32)
    w_fp32 = w_fp32 - grad  # FP32 accumulation

print(f'Initial value: 1.0, train for {n_steps} steps, each step subtracts {lr}')
print(f'Theoretical final value: {1.0 - n_steps * lr:.4f}')
print(f'BF16 master weight: {w_bf16.item():.6f}')
print(f'FP32 master weight: {w_fp32.item():.6f}')
print()
print('Key observation: the BF16 master barely moved (1e-4 < BF16 eps=0.008, all updates lost).')
print('This example shows how FP32 master weights preserve small updates.')


### 3.3 BF16 and FP32 Loss Curves

Run the TinyLM defined below for **300 steps** in each precision and compare the loss curves. This is a small character-level-style model using random input/target IDs, not the repository's nanoGPT training run.

The curves may remain close, illustrating the numerical effect in this particular toy setup. Random targets make this a precision experiment, not a demonstration of useful language learning or final model quality.

The code selects CUDA when available and otherwise CPU. Runtime and BF16 speed depend on the device; the original 1–2 minute estimate is only a rough expectation.


In [ ]:
# BF16 vs FP32 training comparison (mini experiment, runs on both CPU and GPU)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

# Minimal char-level language model, using only Linear + Embedding + hand-written attention
# Does not depend on nn.TransformerEncoder (behavior varies across PyTorch versions)
class TinyCausalAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        b, t, d = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.d_k)
        q, k, v = qkv.unbind(dim=2)  # each [b, t, n_head, d_k]
        q = q.transpose(1, 2)  # [b, n_head, t, d_k]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # causal mask
        mask = torch.tril(torch.ones(t, t, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        h = attn @ v  # [b, n_head, t, d_k]
        h = h.transpose(1, 2).contiguous().view(b, t, d)
        return self.out(h)

class TinyBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.attn = TinyCausalAttention(d_model, n_head)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x))))
        return x

class TinyLM(nn.Module):
    def __init__(self, vocab_size=64, d_model=64, n_layer=2, n_head=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([TinyBlock(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        b, t = x.shape
        pos = torch.arange(t, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

torch.manual_seed(42)
vocab_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
print(f'Running device: {device}')

# Random data (teaching demo; does not depend on shakespeare corpus download)
def get_batch(batch_size=16, seq_len=32):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

def train_one(dtype_label, n_steps=300):
    torch.manual_seed(42)
    model = TinyLM().to(device)
    # FP32 master weight is always kept; BF16 mode relies on autocast
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
    losses = []
    use_amp = dtype_label != 'fp32'
    amp_dtype = torch.bfloat16 if dtype_label == 'bf16' else torch.float16
    t0 = time.time()
    for step in range(n_steps):
        x, y = get_batch()
        optim.zero_grad()
        if use_amp:
            with torch.autocast(device_type=dev_type, dtype=amp_dtype):
                _, loss = model(x, y)
        else:
            _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        if step % 30 == 0 or step == n_steps - 1:
            losses.append((step, loss.item()))
    elapsed = time.time() - t0
    return losses, elapsed

print('Training FP32 ...')
losses_fp32, t_fp32 = train_one('fp32', n_steps=300)
print(f'  done in {t_fp32:.1f}s')
print('Training BF16 (autocast) ...')
losses_bf16, t_bf16 = train_one('bf16', n_steps=300)
print(f'  done in {t_bf16:.1f}s')

print()
print(f'{"step":<8}{"FP32 loss":<14}{"BF16 loss":<14}')
for (s, l1), (_, l2) in zip(losses_fp32, losses_bf16):
    print(f'{s:<8}{l1:<14.4f}{l2:<14.4f}')
print()
print(f'FP32 total time: {t_fp32:.1f}s, BF16 total time: {t_bf16:.1f}s')
print('Compare loss and time in this random-target experiment; speed depends on the device.')


In [ ]:
# Plot the loss curve comparison
import matplotlib.pyplot as plt

steps_fp32 = [s for s, _ in losses_fp32]
vals_fp32 = [l for _, l in losses_fp32]
steps_bf16 = [s for s, _ in losses_bf16]
vals_bf16 = [l for _, l in losses_bf16]

plt.figure(figsize=(8, 4.5))
plt.plot(steps_fp32, vals_fp32, marker='o', label='FP32 (master + compute)', linewidth=2)
plt.plot(steps_bf16, vals_bf16, marker='s', label='BF16 (autocast)', linewidth=2)
plt.xlabel('training step')
plt.ylabel('loss')
plt.title('BF16 vs FP32 training loss (random data)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('This plot compares precision paths for the toy random-target model.')
print('It does not establish final quality on a real language task.')


## 4. FP8 Training

FP8 reduces the data width again. Hopper GPUs provide native FP8 Tensor Core operations, and the DeepSeek-V3 report describes a large-scale FP8 training recipe.

With eight bits, both E4M3 and E5M2 have much tighter range and precision than BF16. Scaling maps tensor values into the representable range.

Scaling can be per-tensor or finer-grained. DeepSeek-V3 uses **per-tile/per-block scaling** to limit the impact of outliers. We examine the variants, this motivation, and implementation details. Reported whole-run costs should not be attributed to FP8 alone without a matched baseline.


### 4.1 E4M3 and E5M2

Two common FP8 variants make different tradeoffs:

| Variant | Exponent bits | Mantissa bits | Finite range | Precision | Common hybrid use |
|:---|:---|:---|:---|:---|:---|
| E4M3 | 4 | 3 | $\pm448$ | Higher | Forward weights and activations |
| E5M2 | 5 | 2 | $\pm57344$ | Lower | Backward gradients |

Forward values often benefit from finer spacing, while gradients may span a wider range. A hybrid recipe uses E4M3 for Forward and E5M2 for Backward; suitable hardware supports both.

This is a recipe choice, not a requirement for all FP8 training. DeepSeek-V3 instead reports E4M3 for all quantized tensors, enabled by fine-grained scaling. Gradients still need sufficient precision as well as range.


In [ ]:
# Look at the ranges of the two FP8 variants (PyTorch 2.1+ supports float8_e4m3fn / e5m2)
import torch

if hasattr(torch, 'float8_e4m3fn'):
    fp8_dtypes = [
        ('FP8 E4M3', torch.float8_e4m3fn),
        ('FP8 E5M2', torch.float8_e5m2),
    ]
    print('=== Numeric info of the two FP8 variants ===')
    for name, dt in fp8_dtypes:
        info = torch.finfo(dt)
        print(f'{name}: max={info.max:<10.1f}  min={info.min:<10.1f}  '
              f'tiny={info.tiny:<10.2e}  eps={info.eps:.4f}')
else:
    print('Current PyTorch does not support FP8 dtypes (needs 2.1+), using a table instead:')
    print('  FP8 E4M3: max=448,    tiny=2^-6=0.0156, eps=0.125')
    print('  FP8 E5M2: max=57344,  tiny=2^-14=6e-5,  eps=0.25')

print()
print('=== Compare with BF16 / FP16 ===')
for dt in [torch.bfloat16, torch.float16]:
    info = torch.finfo(dt)
    print(f'{str(dt):<20}: max={info.max:<12.1f} tiny={info.tiny:<12.2e} eps={info.eps:.4f}')
print()
print('Key observation: FP8 range is much smaller than BF16 (448 vs 3.4e+38).')
print('FP8 needs a suitable scaling recipe; local scales can limit outlier effects.')
print('Per-tensor recipes also exist; validate the choice on the data distribution.')


### 4.2 DeepSeek-V3 Fine-Grained Quantization

The report divides activations into $1\times128$ tiles and weights into $128\times128$ blocks, each with its own scale.

Why use smaller groups? An activation outlier tens of times larger than its neighbors can set an overly broad shared scale. Ordinary values then occupy fewer representable levels. Local scales confine this effect to the outlier's group, preserving resolution elsewhere.

Scales also cost storage and implementation work. Activation tiles have one scale per 128 elements; weight blocks have one per $128\times128$ elements. Kernels must apply these scales while combining partial products. The next code illustrates the grouping idea with **uniform integer quantization**, not the nonuniform E4M3 format.


In [ ]:
# Demonstrate the precision difference of per-tensor vs per-tile scaling
import torch
import numpy as np

torch.manual_seed(0)
# Simulate an activation with a few outliers
x = torch.randn(128) * 0.5
x[5] = 8.0   # outlier
x[60] = -7.0

# Uniform symmetric integer quantization; not an exact E4M3 codebook
def quantize_dequantize(x, scale, n_levels=127):
    """Symmetric quantization + dequantization"""
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# Option 1: per-tensor (one shared scale for the whole segment)
scale_per_tensor = x.abs().max().item() / 127
x_deq_per_tensor = quantize_dequantize(x, scale_per_tensor)

# Option 2: per-tile (split into 128 groups? here we use 8 groups for the demo, 16 elements each)
# Real DeepSeek-V3 uses 1x128; we use a smaller tile here to make the difference clearer
n_tiles = 8
tile_size = len(x) // n_tiles
x_deq_per_tile = torch.zeros_like(x)
for i in range(n_tiles):
    chunk = x[i*tile_size:(i+1)*tile_size]
    s = chunk.abs().max().item() / 127
    x_deq_per_tile[i*tile_size:(i+1)*tile_size] = quantize_dequantize(chunk, s)

# Compare errors
err_per_tensor = (x - x_deq_per_tensor).abs().mean().item()
err_per_tile = (x - x_deq_per_tile).abs().mean().item()

# Look at the error of normal elements, excluding outliers
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60]] = False
err_normal_tensor = (x[mask] - x_deq_per_tensor[mask]).abs().mean().item()
err_normal_tile = (x[mask] - x_deq_per_tile[mask]).abs().mean().item()

print(f'Average error across the whole segment:')
print(f'  per-tensor: {err_per_tensor:.4f}')
print(f'  per-tile (8 groups): {err_per_tile:.4f}')
print(f'Average error of normal elements excluding outliers:')
print(f'  per-tensor: {err_normal_tensor:.4f}  <- outliers widen the scale, big precision loss for normal elements')
print(f'  per-tile:   {err_normal_tile:.4f}  <- outliers affect only their own segment')
print()
print('Key observation: per-tile confines the damage of outliers to the tile they live in,')
print('other tiles retain more integer-level resolution, illustrating local scaling.')


### 4.3 DeepGEMM and High-Precision Accumulation

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) provides efficient low-precision GEMMs. Separate input, accumulator, and output precision:

- **FP8 inputs do not imply BF16 accumulation.** The V3 report uses periodic promotion of partial products into FP32 registers to limit accumulation error. BF16 can be the output dtype; it is not the claimed accumulator format.
- **Scales are applied inside the kernel.** Scale tensors still exist separately, while fusing their application avoids a separate full-tensor dequantization pass.
- **Runtime specialization:** JIT compilation generates kernels suited to matrix shapes. The compilation backend and APIs depend on the checked-out version.
- **TMA and Warp Specialization:** On Hopper, asynchronous transfers and specialized warps overlap data movement and compute.

These techniques improve GEMM efficiency. The V3 report's overall training cost includes many other system choices; it does not establish a universal 50% whole-run saving caused by FP8 alone.


### 4.4 FP8 Training Evidence

Separate published evidence from a proposed local experiment.

**DeepSeek-V3** (2024, MoE, 671B total / 37B active parameters):

- Selected Forward and Backward GEMMs use FP8 with fine-grained scaling.
- Quantized tensors use E4M3; partial accumulation is promoted to FP32.
- Validation runs report relative training-loss differences below 0.25% against BF16. This is a loss comparison, not a claim that every benchmark differs by less than 0.5%.

**Llama-series fine-tuning:** A reader can evaluate FP8 with Transformer Engine on a supported setup. This appendix does not supply an official Meta result establishing 30–40% memory savings or 1.5× throughput; measure those quantities in the chosen reproduction.

**Qwen2.5 / Qwen3:** Do not infer an undisclosed pretraining dtype from community fine-tuning recipes. Consult each official report.

**Hardware:** Native FP8 support depends on the GPU and framework. Transformer Engine supports appropriate Ada, Hopper, and Blackwell devices; A100/V100 lack their native FP8 Tensor Core path. Merely constructing a PyTorch FP8 tensor does not guarantee accelerated training.


## 5. FP4 / INT4 Training

Four-bit training requires different treatment from ordinary BF16. Distinguish training all model parameters in low precision from fine-tuning adapters over a frozen quantized base.

Three examples frame the discussion: QLoRA's widely used four-bit adapter fine-tuning, BitNet's native ternary-weight training, and the stability and kernel challenges of lower-precision pretraining.


### 5.1 QLoRA, NF4, and Double Quantization

QLoRA (Dettmers et al., 2023) keeps a pretrained base frozen in four-bit storage and trains small LoRA adapters using higher-precision compute. The paper demonstrates 65B fine-tuning on a **single 48GB GPU**, rather than 24GB.

Its three main techniques are:

- **NF4 (NormalFloat 4-bit):** A nonuniform 16-level codebook designed for approximately normally distributed weights. More resolution is allocated near zero. It is not ordinary INT4. The symmetric integer example below uses `[-7,…,7]`, which has 15 levels; its normal-quantile codebook illustrates NF4's motivation rather than reproducing the exact production codebook.
- **Double Quantization:** Quantize the quantization constants themselves to reduce scale-storage overhead. This matters when small groups, such as 64 weights, require many scales.
- **Paged Optimizers:** Use unified-memory paging to manage optimizer-memory spikes. This is not the same as keeping every Adam state permanently on the CPU.

QLoRA has been applied to Llama and Qwen families with quality evaluated against higher-precision fine-tuning. It adapts a pretrained base; it does not demonstrate training every base weight from scratch in four-bit storage. Adapter dtype and compute dtype depend on the implementation.


In [ ]:
# Demonstrate the core idea of NF4: quantile-based quantization vs uniform quantization
import numpy as np
from scipy.stats import norm

np.random.seed(42)
# Simulate a normally distributed weight
w = np.random.randn(10000).astype(np.float32)

# Symmetric integer example: 15 levels in [-7, 7]
def int4_quant(w):
    scale = np.max(np.abs(w)) / 7
    q = np.round(w / scale).clip(-7, 7)
    return q * scale

# NF4 motivation: 16 normal-quantile levels, not the exact production codebook
# quantile = norm.ppf((i + 0.5) / 16) for i in 0..15, then normalize to [-1, 1]
nf4_levels = norm.ppf((np.arange(16) + 0.5) / 16)
nf4_levels = nf4_levels / np.max(np.abs(nf4_levels))  # normalize to [-1, 1]

def nf4_quant(w):
    scale = np.max(np.abs(w))  # NF4 assumes the weights are already normalized
    w_norm = w / scale
    # Find the nearest NF4 level for each weight
    q = np.zeros_like(w)
    for i, val in enumerate(w_norm):
        idx = np.argmin(np.abs(nf4_levels - val))
        q[i] = nf4_levels[idx]
    return q * scale

w_int4 = int4_quant(w)
w_nf4 = nf4_quant(w)

mae_int4 = np.mean(np.abs(w - w_int4))
mae_nf4 = np.mean(np.abs(w - w_nf4))

print(f'Weight distribution: normal(0, 1), 10000 samples')
print(f'INT4 uniform quantization MAE: {mae_int4:.4f}')
print(f'NF4 quantile quantization MAE: {mae_nf4:.4f}  <- usually lower')
print()
print('Key observation: NF4 clusters its 16 levels near 0 (where the normal density is high),')
print('which fits the typical weight distribution better. This is why QLoRA uses NF4 instead of INT4.')


### 5.2 BitNet 1.58-bit Pretraining

BitNet b1.58 uses ternary linear weights $\{-1,0,+1\}$, giving an ideal information budget of $\log_2 3\approx1.58$ bits per weight. Actual storage also includes packing and scale overhead.

Its main ideas are:

- Ternary weights allow specialized matrix multiplication based on addition, subtraction, and skipping zero values.
- The b1.58 recipe quantizes activations to **eight-bit integers**, rather than leaving them unquantized in BF16.
- A Straight-Through Estimator (STE) passes surrogate gradients through the nondifferentiable quantization operation; training retains higher-precision latent weights.

The 2024 paper compares quality and efficiency with FP16 baselines at matched scales. The later [BitNet b1.58 2B4T report](https://arxiv.org/abs/2504.12285) releases a native low-bit model. Stability, kernels, and integration remain important; the existence of a released model should not be confused with universal adoption.


### 5.3 Challenges of Low-Bit Pretraining

QLoRA demonstrates adapter fine-tuning over a quantized base; BitNet demonstrates a different native low-bit architecture. Neither makes all four-bit pretraining recipes interchangeable.

**Stability:** Quantization error can affect early training, produce loss spikes, or alter optimization. Higher-precision warmup followed by lower precision is one possible recipe, not a universally required or sufficient fix.

**Kernels:** BF16 has mature libraries; FP8 has implementations such as Transformer Engine and DeepGEMM. FP4 support is also developing, but availability depends on hardware, layout, and training recipe. Check the framework's actual supported operations rather than assuming weight compression enables the entire training loop.

**Total memory:** Activations and optimizer states can dominate training storage. Reducing weight width does not automatically compress these components. Their sensitivity and accumulation needs may require higher precision, so total savings depend on the inventory.

BF16 is a common baseline, FP8 and FP4 require validated recipes, and QLoRA remains an adapter fine-tuning method. Inference INT4/INT8 kernels and training kernels solve different problems.


## 6. Common Mixed-Precision Problems

The following checklist collects issues encountered across the course. Separate common Dense/MoE problems from routing-specific problems, and examine symptom → possible cause → diagnostic steps.


### 6.1 Problems Shared by Dense and MoE Models

**1. Gradients vanish or underflow**

- **Symptom:** Loss stops improving and some gradient tensors have many zeros.
- **Possible cause:** FP16 gradients become subnormal or round to zero; low-precision accumulation can also lose small changes. Values below $6\times10^{-5}$ are not automatically zero.
- **Check:** Log gradient min/max and nonzero fractions. Use Loss Scaling for FP16; inspect compute and accumulation dtypes for BF16 rather than assuming more autocast coverage fixes everything.

**2. Attention probabilities saturate**

- **Symptom:** Many Attention weights approach 0 or 1.
- **Possible cause:** Large Q/K magnitudes, missing scale, or range/precision problems. Saturation alone does not prove overflow, and sequence length alone is not the cause.
- **Check:** Inspect score ranges, scaling, masks, stable Softmax, and kernel accumulator precision. Compare against an FP32 reference or a supported FlashAttention path.

**3. Optimizer-state rounding**

- **Symptom:** Loss stalls or worsens after many steps.
- **Possible cause:** Repeated low-precision state updates lose information.
- **Check:** Inspect optimizer-state dtypes. Use FP32 as a robust baseline, or a validated low-precision optimizer rather than casting states indiscriminately.

**4. Micro-batch gradient accumulation**

- **Symptom:** Gradient accumulation behaves differently from the equivalent large batch.
- **Possible cause:** Low-precision accumulation, loss-normalization differences, or other batch-dependent behavior.
- **Check:** Verify normalization and the accumulation buffer's dtype. FP32 accumulation can help; `zero_grad(set_to_none=True)` alone does not select a buffer dtype.


### 6.2 MoE-Specific Problems

Routing is sensitive to small score differences.

**1. Routing precision**

Dead experts or unstable assignments can have many causes, including rounded gating scores. Keeping router scores and probability calculations in FP32 is a useful baseline when supported by the architecture. Disable autocast for that path explicitly if needed, and inspect top-k assignments. It is a recommended diagnostic, not a universal requirement for every MoE implementation.

**2. Auxiliary-loss precision**

A fluctuating or ineffective load-balancing loss may involve poorly accumulated probabilities or statistics. Compute sensitive reductions in FP32 and use suitable count/statistic buffers. Expert counts can be integer-valued; probability sums need sufficient precision.

**3. Imbalanced loads**

If a few experts receive over 80% of tokens, inspect token fractions, routing scores, capacity limits, and balancing objectives. Precision can amplify near-tie differences, but cannot be assumed to be the sole cause. Depending on the architecture, adjust the auxiliary-loss weight or its separate balancing mechanism.

**4. Shared and routed experts**

Models with shared experts require separate inspection of both paths. Different dtypes can affect numerical behavior, but do not by themselves prove that shared experts receive extra routed tokens or duplicate learned features. Compare precision policies, outputs, and gradients before attributing a failure to this difference.


## 7. Practical Guide

Choose precision according to the phase—Pretraining, Fine-Tuning, or Inference—and the architecture, Dense or MoE. The following table of choices gives starting points to validate on the target hardware.


### 7.1 Choosing Precision

```text
Pretraining from scratch
├── Dense
│   ├── Supported FP8 GPU: FP32 master + validated FP8 compute recipe
│   └── A100: BF16 autocast with suitable higher-precision state
└── MoE
    ├── Large-scale validated setup: FP8 compute with suitable accumulation
    └── Baseline reproduction: BF16, inspect router precision separately

Fine-tuning
├── Full parameters: BF16 compute with suitable optimizer/master precision
├── LoRA: BF16 base compute; adapter/master dtype follows the framework
└── QLoRA: frozen NF4 base; higher-precision compute and adapters

Inference
├── Dense: INT4/INT8 weights; FP16/BF16 activations where supported
└── MoE: supported quantized weights; inspect routing precision separately
```

Pretraining emphasizes stable optimization, adapter fine-tuning often emphasizes memory, and inference balances throughput, latency, and quality. These are starting configurations, not a guarantee that every model and kernel supports each combination. Validate the router separately in MoE.


### 7.2 Verifying Memory Savings

BF16 autocast does not halve every component. FP32 parameters and Adam states can remain unchanged. The following is an illustrative inventory, not a universal implementation:

| Component | FP32 baseline | BF16 autocast example | FP8 recipe example |
|:---|:---|:---|:---|
| Master weights | FP32, 4 bytes/parameter | FP32, 4 bytes | FP32, 4 bytes |
| Compute weight copy | None | BF16, possibly cached | FP8 plus scales, recipe-dependent |
| Activations | FP32 | Selected tensors BF16 | Selected tensors FP8 plus scales |
| Parameter gradients | FP32 | FP32 for FP32 parameters | Recipe-dependent; inspect accumulation |
| Adam states | Two FP32 states, 8 bytes | Two FP32 states, 8 bytes | Recipe-dependent; FP32 is a baseline |

Savings depend on saved activations, copies, scales, and peaks. The code below assumes activation element count equals parameter count for teaching. It estimates **20P bytes versus 18P bytes, a 10% saving**, and excludes compute copies and workspace. It does not measure actual CUDA peak memory or establish a general 30–40% saving.


In [ ]:
# Teaching estimate: compare the memory estimates of "full FP32 training" vs "BF16 autocast training"
import torch
import torch.nn as nn

def estimate_memory(model, mode='fp32'):
    """
    Estimate memory usage under two training modes.
    mode='fp32': full FP32 training (weight/grad/adam/activation all FP32)
    mode='bf16': BF16 autocast (master/grad/adam still FP32, activation is BF16)
    """
    n = sum(p.numel() for p in model.parameters())

    # master weight: FP32 in both modes
    master_weight = n * 4

    # gradient: accumulated to FP32 in both modes
    gradient = n * 4

    # Adam state: FP32 in both modes (momentum + variance)
    adam_state = n * 8

    # activation assumption: roughly equal to model parameter count (teaching simplification)
    if mode == 'fp32':
        activation = n * 4   # FP32 activation
    else:  # bf16
        activation = n * 2   # BF16 activation

    total = master_weight + gradient + adam_state + activation

    label = 'Full FP32 training' if mode == 'fp32' else 'BF16 autocast'
    print(f'\n=== {label} ({n/1e6:.1f}M params) ===')
    print(f'master weight (FP32):  {master_weight/1e9:.3f} GB')
    print(f'gradient (FP32):       {gradient/1e9:.3f} GB')
    print(f'Adam state (FP32x2):   {adam_state/1e9:.3f} GB')
    print(f'activation:            {activation/1e9:.3f} GB')
    print(f'total:                 {total/1e9:.3f} GB')
    return total

# Estimate with a 100M parameter model
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1000, 100000)

model = DummyModel()
n = sum(p.numel() for p in model.parameters())
print(f'Model parameter count: {n/1e6:.1f}M')

t_fp32 = estimate_memory(model, 'fp32')
t_bf16 = estimate_memory(model, 'bf16')

saved = (1 - t_bf16 / t_fp32) * 100
print(f'\nBF16 saves {saved:.1f}% compared to full FP32')
print('Key observation: master weight + gradient + Adam state remain unchanged in this example,')
print('so the memory benefit of low-precision training comes mainly from activation.')


### 7.3 NaN Diagnostic Checklist

Inspect these in order:

1. **Loss Scaling:** For FP16, check nonfinite-gradient detections and skipped updates. Many consecutive skips can indicate an unsuitable scale or unstable computation; occasional skips are not mandatory for healthy training.
2. **Gradients:** After Backward, inspect per-parameter min/max/mean and locate the first Inf/NaN layer. Unscale before interpreting scaled-gradient magnitudes.
3. **Attention:** Check Q/K magnitudes, masks, and scaling. Very sharp Softmax can have small gradients; large scores do not automatically imply exploding gradients.
4. **LayerNorm:** Inspect variance, epsilon, and dtype. The denominator involves $\sqrt{\mathrm{variance}+\epsilon}$, not bare variance.
5. **Learning-rate warmup:** An excessive early learning rate can destabilize weights.
6. **Data:** Ensure Token IDs lie within `vocab_size`. Out-of-range Embedding indices generally raise an indexing/device-assert error rather than directly producing NaN.

`torch.autograd.set_detect_anomaly(True)` can report the Forward operation responsible for a nonfinite Backward result. It slows training; the overhead depends on the workload.


## Appendix: FP8 Engineering Details

The earlier sections explain formats and scaling. This section provides a source-reading route and a small environment example for readers who want to investigate real FP8 kernels and fine-tuning.


### A3.1 Reading DeepGEMM

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) is a useful source for low-precision GEMM implementation. CUDA and GPU architecture knowledge help. Pin a commit first: the repository has evolved, so historical file paths and API names need not exist on current main.

**First pass: Public APIs and tests**

1. Read `deep_gemm/__init__.py` and its exported dense/grouped GEMM interfaces.
2. Follow the data and scale arguments into the implementation. The historical `deep_gemm/core/__init__.py` path is version-specific; locate the corresponding code in your checkout.
3. Read the GEMM tests under `tests/`; the historical `tests/test_fp8_gemm.py` name may differ by version.

Ask: Are scales per-tensor, per-tile, or per-block? How do grouped GEMM inputs differ from ordinary GEMM inputs?

**Second pass: Compilation and kernels**

4. Follow the runtime compilation entry point. `deep_gemm/jit_kernels.py` describes a historical layout; current layouts can differ.
5. Inspect CUDA headers and source under `csrc/`, locating the relevant GEMM kernel.

Look for three things:

- **Scale storage:** Data and scales are separate tensors with matching logical groups. For a weight matrix `[K,N]` and $128\times128$ groups, the logical scale grid is `[ceil(K/128),ceil(N/128)]`; orientation, padding, and physical layout depend on the API.
- **Accumulation:** Distinguish Tensor Core partial accumulators, promotion to FP32, and the final BF16 output. Do not infer BF16 accumulation from the output dtype.
- **Launch structure:** Blocks compute output tiles, such as $128\times128$, and the grid covers the output. On Hopper, TMA loads tiles asynchronously while specialized warps overlap loading and computation.

**Third pass: Grouped GEMM for MoE**

6. Find the grouped GEMM interfaces and tests; historical `deep_gemm/grouped_gemm.py` paths are version-specific.
7. Follow the handling of variable token counts per expert into the matching kernels.

Launching a separate small GEMM for each expert incurs overhead. Grouped GEMM combines the work into fewer launches, grouping tokens by expert. Experts generally share the matrix width while their token dimension varies.

**Prerequisites:** CUDA grids/blocks/threads, shared memory and warps; Hopper TMA and Warp Specialization; and the basic matrix abstractions of CUTLASS/CuTe. Verify current hardware and CUDA requirements in the pinned repository before building.


### A3.2 A Small FP8 Training Setup

Begin with a supported Linear operation before attempting Llama fine-tuning. A full large-scale V3 pretraining run is outside a single-GPU reproduction; Llama fine-tuning capacity depends on model size, sequence length, adapters, and sharding.

**Environment**

```text
Hardware: a Transformer Engine supported Ada, Hopper, or Blackwell GPU
CUDA/cuDNN/PyTorch/Python: match the pinned Transformer Engine release
Memory: measure the complete model, optimizer, activations, and workspace
```

Consult the [installation requirements](https://docs.nvidia.com/deeplearning/transformer-engine/installation.html) rather than assuming any CUDA 12.x combination works.

**Dependencies**

```bash
# Quote extras so shells do not expand brackets
pip install 'transformer_engine[pytorch]'

# Optional attention library; installing it alone does not enable FP8 attention
pip install flash-attn --no-build-isolation

# Optional training frameworks; their FP8 integration must be configured
pip install trl
# or
pip install megatron-core
```

**A minimal operation (Transformer Engine 2.x API)**

```python
import torch
import transformer_engine.pytorch as te
from transformer_engine.common.recipe import DelayedScaling, Format

model = te.Linear(4096, 4096, params_dtype=torch.float32).cuda()
input = torch.randn(16, 4096, device='cuda', dtype=torch.bfloat16)
recipe = DelayedScaling(
    fp8_format=Format.HYBRID,
    margin=0,
    amax_history_len=16,
    amax_compute_algo='max',
)
with te.autocast(enabled=True, recipe=recipe):
    output = model(input)
    loss = output.float().square().mean()
loss.backward()  # Outside the autocast context
```

This demonstrates one Forward/Backward operation, not an entire fine-tuning loop. Older releases use `te.fp8_autocast(fp8_recipe=...)`; consult the matching version. `te.Linear(..., fp8=True)` is not the switch used here.

Important scaling settings include `margin` (larger is more conservative), `amax_history_len` (history window), and `amax_compute_algo` (`max` or `most_recent`). Available recipes and options vary by version.

**Verify the compute path**

```python
# Output dtype alone does not reveal GEMM operand/accumulator precision
hook = model.register_forward_hook(
    lambda m, i, o: print(f'{m.__class__.__name__} output dtype: {o.dtype}')
)

# Profile the actual training script:
# nsys profile -t cuda,nvtx python train.py
```

Inspect supported recipe diagnostics and the profiler's kernel/operand details. Custom kernels need not be named `cublasH8Gemm`, and FP8 GEMM can return BF16 output. If the expected path is absent, check context coverage, supported shapes, hardware, and library compatibility. See the [official FP8 setup example](https://docs.nvidia.com/deeplearning/transformer-engine/getting_started/index.html).


## Summary

- [ ] Write the exponent/mantissa allocation of FP32, FP16, BF16, and FP8 E4M3/E5M2, and explain range versus precision.
- [ ] Explain why BF16 usually does not require FP16-style Loss Scaling.
- [ ] Explain how FP32 master weights preserve repeated small updates.
- [ ] Check `torch.autocast` policies for the target backend rather than assuming one global dtype list.
- [ ] Distinguish a hybrid E4M3/E5M2 recipe from fine-grained all-E4M3 training.
- [ ] Separate FP8 inputs, high-precision accumulation, and BF16 output in GEMMs.
- [ ] Explain QLoRA's NF4, Double Quantization, and Paged Optimizers.
- [ ] Diagnose routing and auxiliary-loss precision separately in MoE.
- [ ] Choose and validate a precision recipe for each training or inference scenario.


## Exercises

Precision becomes easier to understand after calculating it yourself. These three exercises revisit the core concepts.

> You can ask AI for an approach, a breakdown, or a check, but avoid having it finish the whole exercise.

**Exercise 1: BF16 master-weight accumulation**

Start with $w=1.0$ and subtract $g=0.0001$ for **1000 steps**, in BF16 and FP32. Compare both final values with the theoretical result.

Hint: BF16 spacing near 1 is about 0.008, much larger than this update.


In [ ]:
# Exercise 1: BF16 vs FP32 master weight accumulation
import torch

n_steps = 1000
grad = 1e-4

# TODO: accumulate in BF16 for n_steps steps
w_bf16 = torch.tensor([1.0], dtype=torch.bfloat16)
for _ in range(n_steps):
    w_bf16 -= grad

# TODO: accumulate in FP32 for n_steps steps
w_fp32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(n_steps):
    w_fp32 -= grad

# TODO: compute the theoretical final value
w_theoretical = 1.0 - n_steps * grad

assert w_bf16 is not None and w_fp32 is not None and w_theoretical is not None
assert abs(w_fp32.item() - w_theoretical) < 1e-4, f'FP32 should be close to the theoretical value, diff {abs(w_fp32.item() - w_theoretical)}'
print(f'Theoretical final value:  {w_theoretical:.6f}')
print(f'BF16 accumulation result: {w_bf16.item():.6f}')
print(f'FP32 accumulation result: {w_fp32.item():.6f}')
print(f'BF16 deviation from theory: {abs(w_bf16.item() - w_theoretical):.6f}')
print(chr(10004) + 'TODO: replace this placeholder with your code')

**Exercise 2: Local scaling and quantization error**

Create 128 elements with three outliers. Compare per-tensor and per-tile (tile_size=16) **four-bit symmetric integer quantization**, then calculate mean error on ordinary elements after excluding the outliers. This illustrates scaling locality, rather than implementing E4M3 FP8.

Hint: Reshape to `[n_tiles,tile_size]` and calculate a separate scale for each row.


In [ ]:
# Exercise 2: per-tensor vs per-tile scaling
import torch

torch.manual_seed(42)
x = torch.randn(128) * 0.5
x[5], x[60], x[100] = 8.0, -7.0, 6.5  # 3 outliers

def quantize_dequantize(x, scale, n_levels=7):  # 4-bit: levels = 7
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# TODO: per-tensor quantization (one shared scale for the whole segment)
scale_tensor = x.abs().max() / 7
x_deq_tensor = quantize_dequantize(x, scale_tensor)

# TODO: per-tile quantization (tile_size=16, 8 tiles in total)
tile_size = 16
x_deq_tile = torch.zeros_like(x)
for i in range(x.numel() // tile_size):
    tile = x[i * tile_size:(i + 1) * tile_size]
    scale_tile = tile.abs().max() / 7
    x_deq_tile[i * tile_size:(i + 1) * tile_size] = quantize_dequantize(tile, scale_tile)

assert scale_tensor is not None and x_deq_tensor is not None
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60, 100]] = False
err_tensor = (x[mask] - x_deq_tensor[mask]).abs().mean().item()
err_tile = (x[mask] - x_deq_tile[mask]).abs().mean().item()

print(f'per-tensor normal-element MAE: {err_tensor:.4f}')
print(f'per-tile  normal-element MAE: {err_tile:.4f}')
assert err_tile < err_tensor, 'per-tile should make the error of normal elements smaller (outliers affect only their own segment)'
print(chr(10004) + ' Exercise 2 passed: per-tile scaling is a key engineering technique for accurate FP8 training')


**Exercise 3: Estimate BF16 versus FP32 memory savings**

Compare full FP32 training (weights, gradients, Adam states, and activations) with a BF16-autocast teaching estimate. In the latter, master weights, gradients, and Adam states remain FP32; only the assumed activations become BF16.

Use **1B parameters**, with **1B activation elements** for this simplified inventory. Exclude temporary compute copies and workspace explicitly.

Hint: Master weights cost 4 bytes/parameter, gradients 4, Adam states 8. Activations change from 4 bytes/element to 2.


In [ ]:
# Exercise 3: estimate training memory of a 1B model
# Convention: here we compare "full FP32 training" vs "BF16 autocast training" vs "FP8 training"
# Full FP32 training: weight, gradient, Adam state all FP32
# BF16/FP8 training: master weight + gradient + Adam state still FP32, only activation is saved

n_params = 1_000_000_000  # 1B

# Scenario A: full FP32 training (weight + gradient + adam all FP32)
# Hint: weight/gradient each 4B/param, Adam state (momentum+variance) 8B/param
weight_fp32_bytes = 4 * n_params
gradient_fp32_bytes = 4 * n_params
adam_state_bytes = 8 * n_params

# Scenario B: BF16 autocast training
# master weight is still FP32, gradient accumulates to FP32, Adam state still FP32
# Only difference: activation is stored as BF16 (2B/element) instead of FP32 (4B/element) during forward
# Assume activation total is equivalent to 1B params (teaching simplification)
activation_fp32_bytes = 4 * n_params
activation_bf16_bytes = 2 * n_params

assert all(v is not None for v in [
    weight_fp32_bytes, gradient_fp32_bytes, adam_state_bytes,
    activation_fp32_bytes, activation_bf16_bytes
])

# Full FP32 training memory: weight + grad + adam + activation all FP32
total_fp32 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_fp32_bytes
# BF16 autocast：master FP32 + grad FP32 + adam FP32 + activation BF16
total_bf16 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_bf16_bytes

print(f'1B model training memory estimate (including activation):')
print(f'  Full FP32 training:        {total_fp32/1e9:.2f} GB')
print(f'  BF16 autocast training:    {total_bf16/1e9:.2f} GB (saves {(1-total_bf16/total_fp32)*100:.1f}%)')

assert total_bf16 < total_fp32, 'BF16 should save activation memory'
print(chr(10004) + ' Exercise 3 passed: master weights and optimizer state remain high precision, so low precision mainly saves activation memory')


## References

- Micikevicius et al., [Mixed Precision Training](https://arxiv.org/abs/1710.03740), 2017 — FP16 training, Loss Scaling, and master weights.
- Kalamkar et al., [A Study of BF16 for Deep Learning](https://arxiv.org/abs/1905.12322), 2019 — BF16 analysis.
- DeepSeek-AI, [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437), 2024 — Fine-grained FP8 scaling and high-precision accumulation.
- DeepSeek-AI, [DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) — Low-precision GEMM implementation.
- NVIDIA, [Transformer Engine](https://github.com/NVIDIA/TransformerEngine) — Low-precision training library.
- Micikevicius et al., [FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433), 2022 — E4M3 and E5M2 formats.
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023 — NF4, Double Quantization, and Paged Optimizers.
- Wang et al., [BitNet: Scaling 1-bit Transformers for Large Language Models](https://arxiv.org/abs/2310.11453), 2023 — Original BitNet.
- Ma et al., [The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits](https://arxiv.org/abs/2402.17764), 2024 — BitNet b1.58.
